In [1]:
# This file introduces a new algorithm for accelerating Algorithm 4 (MMJ distance by Calculation and Copy)
# by parallel computing, the new algorithm is called Algorithm 13 (APPD accelerated by parallel computing).


In [2]:
import time
import pickle
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import pairwise_distances
import networkx as nx
import sys
from joblib import Parallel, delayed
import random
from numba import njit
import threading

In [3]:
 
# test_data_145 = pickle.load( open( "./data/test_data_145.p", "rb" ) ) 
 

In [4]:
@njit(parallel=True, cache=True, fastmath=True)
def primMST_numba(distance_matrix):
    V = distance_matrix.shape[0]
    key = np.full(V, np.inf)
    parent = -np.ones(V, dtype=np.int64)
    inMST = np.zeros(V, dtype=np.bool_)
    key[0] = 0.0
    
    for _ in range(V):
    
        u = -1
        min_val = np.inf
        for v in range(V):
            if (not inMST[v]) and (key[v] < min_val):
                min_val = key[v]
                u = v
        if u == -1:
            break
        inMST[u] = True
        for v in range(V):
        
            if (distance_matrix[u, v] > 0) and (not inMST[v]) and (distance_matrix[u, v] < key[v]):
                key[v] = distance_matrix[u, v]
                parent[v] = u
    return parent

def construct_MST_from_graph(distance_matrix):
    V = distance_matrix.shape[0]

    parent = primMST_numba(distance_matrix)
    
    MST = nx.Graph()
    for i in range(V):
        MST.add_node(i)

    for i in range(1, V):
        MST.add_edge(parent[i], i, weight=distance_matrix[i, parent[i]])
    return MST

In [5]:
# n_processors is the number of processors.

def cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu_algo_13(distance_matrix, n_processors):
    global mmj_matrix  
    lenX = len(distance_matrix)
    
    # distance_matrix = np.round(pairwise_distances(X), round_n)
    
    mmj_matrix = np.zeros((lenX, lenX)) 
 
    
    MST = construct_MST_from_graph(distance_matrix)
 

    MST_edge_list = list(MST.edges(data='weight'))

    edge_node_list = [(edge[0], edge[1]) for edge in MST_edge_list]
    edge_weight_list = [edge[2] for edge in MST_edge_list]
    edge_large_to_small_arg = np.argsort(edge_weight_list)[::-1]
    edge_weight_large_to_small = np.sort(edge_weight_list)[::-1]
    edge_nodes_large_to_small = [edge_node_list[i] for i in edge_large_to_small_arg] 
 
    MST_list = []
    MST_list.append(MST)
    
    for j in range(n_processors - 1):
        MST_copy = MST.copy(as_view=False)
        MST_list.append(MST_copy)

    current_removed_edge_index = [-1]*n_processors
    mst_locked = [False]*n_processors
 
    N = lenX - 1
 
    Parallel(n_jobs = n_processors, backend="threading")(
        delayed(for_parallel_compu)(i, MST_list, edge_nodes_large_to_small, edge_weight_large_to_small, current_removed_edge_index, mst_locked)
        for i in range(N)
    )
 
    return mmj_matrix
 

In [6]:
def for_parallel_compu(i, MST_list, edge_nodes_large_to_small, edge_weight_large_to_small, current_removed_edge_index, mst_locked):
 
    global mmj_matrix
 
    for j in range(n_processors + 1):
        assert j < n_processors, "j < n_processors"
        if not mst_locked[j]:
            if current_removed_edge_index[j] < i:
                temppp = j
                break
 
    MST_temp = MST_list[temppp]
    mst_locked[temppp] = True
 
    P = current_removed_edge_index[temppp] + 1
 
    for kk in range(P, i + 1):
        edge_nodes = edge_nodes_large_to_small[kk]
        MST_temp.remove_edge(*edge_nodes)
        current_removed_edge_index[temppp] += 1
 

    edge_weight = edge_weight_large_to_small[i]

    tree1_nodes = list(nx.dfs_preorder_nodes(MST_temp, source=edge_nodes[0]))
    tree2_nodes = list(nx.dfs_preorder_nodes(MST_temp, source=edge_nodes[1]))
 
    mst_locked[temppp] = False
  
    idx1, idx2 = np.meshgrid(tree1_nodes, tree2_nodes, indexing="ij")
    mmj_matrix[idx1, idx2] = mmj_matrix[idx2, idx1] = edge_weight
    # print(current_removed_edge_index, i, threading.get_ident())

 

In [7]:


def create_symmetric_distance_matrix(n, seed=222, low=1, high=1000):
    if seed is not None:
        np.random.seed(seed)
    A = np.random.randint(low, high, size=(n, n))
    sym_A = (A + A.T) // 2  # Ensure symmetry and integer values
    for k in range(n):
        sym_A[k, k] = 0
    return sym_A.astype('float64')
 


In [8]:

# n_processors is the number of processors.
n_processors = 4

NN = 10000
 
distance_matrix = create_symmetric_distance_matrix(NN)

print(f"Number of points in data X: {NN}" )
 

mmj_matrix = None
 
start = time.time()
X_mmj_matrix_algo_4_parallel_compu = cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu_algo_13(distance_matrix, n_processors)
end = time.time()
time_used = end - start
time_used = np.round(time_used, 3)

print(f"Time used: {time_used}s" )

Number of points in data X: 10000
Time used: 32.121s


In [9]:
X_mmj_matrix_algo_4_parallel_compu[0, -30:]

array([11., 11., 11., 14., 11., 11., 11., 11., 11., 11., 11., 11., 11.,
       11., 11., 13., 11., 11., 11., 11., 11., 11., 11., 11., 11., 11.,
       11., 11., 11., 11.])